# Adaptive Adversarial Threat Detection and Defense Framework
## Notebook 2 — Adversarial Attack Simulation (Week 5)

**Prerequisite:** Run `week-4/notebook-baseline.ipynb` first so that `X_train`, `X_test`,
`y_train`, `y_test`, `rf_baseline`, `top5_idx`, `feature_names`, and baseline metrics
(`acc_c1`, `prec_c1`, `rec_c1`, `f1_c1`, `auc_c1`) are all in memory.

If starting a new Colab session, re-run notebook-baseline.ipynb Cells 1–12 first.

In [ ]:
# ─── Cell 1: Confirm prerequisites are available ──────────────────────────────
# This cell will fail if notebook-baseline.ipynb has not been run first.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)
print('Prerequisites check:')
print(f'  X_test shape:   {X_test.shape}')
print(f'  top5_idx:       {top5_idx}')
print(f'  Baseline recall: {rec_c1:.4f}')

In [ ]:
# ─── Cell 2: Adversarial test-set generator function ──────────────────────────
# Creates a perturbed copy of X_test.
# Only attack-class rows (y_test == 1) are modified.
# Normal-class rows are left untouched.

def create_adversarial_test(X_test, y_test, top5_idx, noise_std, random_state=42):
    """
    Add Gaussian noise to the top-5 features of attack-class test samples.
    
    Parameters
    ----------
    X_test      : numpy array, shape (n_samples, n_features)
    y_test      : numpy array, binary labels (0=normal, 1=attack)
    top5_idx    : list/array of 5 feature column indices to perturb
    noise_std   : standard deviation of Gaussian noise
    random_state: seed for reproducibility
    
    Returns
    -------
    X_adv : perturbed copy of X_test
    """
    np.random.seed(random_state)
    X_adv = X_test.copy()                       # copy — never modify original
    attack_rows = np.where(y_test == 1)[0]       # indices of attack-class rows
    noise = np.random.normal(
        loc=0,
        scale=noise_std,
        size=(len(attack_rows), len(top5_idx))  # shape: (n_attack_rows, 5)
    )
    # np.ix_ builds an open mesh so we can index rows and columns simultaneously
    X_adv[np.ix_(attack_rows, top5_idx)] += noise
    return X_adv

print('create_adversarial_test() defined.')

In [ ]:
# ─── Cell 3: Run attack at 3 noise levels, record metrics ─────────────────────
noise_levels = [0.1, 0.3, 0.5]
attack_results = {}

for std in noise_levels:
    X_adv = create_adversarial_test(X_test, y_test, top5_idx, noise_std=std)
    y_pred_adv = rf_baseline.predict(X_adv)
    y_prob_adv = rf_baseline.predict_proba(X_adv)[:, 1]

    attack_results[std] = {
        'accuracy': accuracy_score(y_test, y_pred_adv),
        'recall':   recall_score(y_test, y_pred_adv),
        'f1':       f1_score(y_test, y_pred_adv),
        'auc':      roc_auc_score(y_test, y_prob_adv)
    }

print('=== ADVERSARIAL ATTACK RESULTS ===')
print(f'{"Noise":>8} | {"Accuracy":>10} | {"Recall":>8} | {"F1":>8} | {"AUC":>8}')
print('-' * 55)
for std, m in attack_results.items():
    print(f'{std:>8.1f} | {m["accuracy"]:>10.4f} | {m["recall"]:>8.4f} | {m["f1"]:>8.4f} | {m["auc"]:>8.4f}')

In [ ]:
# ─── Cell 4: Compare against baseline ─────────────────────────────────────────
print('=== COMPARISON: C1 BASELINE vs. C2 UNDER ATTACK ===')
print(f'{"Condition":>22} | {"Accuracy":>10} | {"Recall":>8} | {"F1":>8}')
print('-' * 58)
print(f'{"C1 Baseline":>22} | {acc_c1:>10.4f} | {rec_c1:>8.4f} | {f1_c1:>8.4f}')
for std, m in attack_results.items():
    label = f'C2 Attack std={std}'
    drop_rec = rec_c1 - m['recall']
    drop_f1  = f1_c1  - m['f1']
    print(f'{label:>22} | {m["accuracy"]:>10.4f} | {m["recall"]:>8.4f} | {m["f1"]:>8.4f}  (recall drop: -{drop_rec:.4f})')

In [ ]:
# ─── Cell 5: Select defense noise level ───────────────────────────────────────
# Default: std=0.3 (medium — meaningful but not total degradation).
# Adjust if your results show:
#   - std=0.3 causes < 0.05 recall drop → use 0.5 instead
#   - std=0.3 causes > 0.50 recall drop → use 0.1 instead

DEFENSE_NOISE_STD = 0.3   # <-- change only if justified by your results above

# Generate the selected adversarial test set (used in Week 6)
X_adv_selected      = create_adversarial_test(X_test, y_test, top5_idx, noise_std=DEFENSE_NOISE_STD)
y_pred_adv_selected = rf_baseline.predict(X_adv_selected)
y_prob_adv_selected = rf_baseline.predict_proba(X_adv_selected)[:, 1]

# Store C2 metrics at the selected level
acc_c2  = accuracy_score(y_test, y_pred_adv_selected)
rec_c2  = recall_score(y_test, y_pred_adv_selected)
f1_c2   = f1_score(y_test, y_pred_adv_selected)
auc_c2  = roc_auc_score(y_test, y_prob_adv_selected)

print(f'Selected defense noise level: std={DEFENSE_NOISE_STD}')
print(f'C2 Accuracy : {acc_c2:.4f}')
print(f'C2 Recall   : {rec_c2:.4f}')
print(f'C2 F1-Score : {f1_c2:.4f}')
print(f'C2 AUC-ROC  : {auc_c2:.4f}')

---
## Visualizations

In [ ]:
# ─── Cell 6: Attack curve — recall + F1 vs. noise level ───────────────────────
noise_levels_plot = [0.0, 0.1, 0.3, 0.5]
recall_values = [rec_c1] + [attack_results[s]['recall'] for s in [0.1, 0.3, 0.5]]
f1_values     = [f1_c1]  + [attack_results[s]['f1']     for s in [0.1, 0.3, 0.5]]

plt.figure(figsize=(8, 5))
plt.plot(noise_levels_plot, recall_values, marker='o', color='crimson',
         linewidth=2, label='Recall (Attack Detection Rate)')
plt.plot(noise_levels_plot, f1_values, marker='s', color='steelblue',
         linewidth=2, linestyle='--', label='F1-Score')
plt.axvline(x=DEFENSE_NOISE_STD, color='gray', linestyle=':',
            label=f'Selected defense level (std={DEFENSE_NOISE_STD})')
plt.xlabel('Gaussian Noise Standard Deviation', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.title('Attack Curve: Model Performance vs. Adversarial Noise Level', fontsize=13)
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('attack-curve.png', dpi=150)
plt.show()
print('Saved: attack-curve.png')

In [ ]:
# ─── Cell 7: Confusion matrix for Condition 2 ─────────────────────────────────
cm_attack = confusion_matrix(y_test, y_pred_adv_selected)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_attack, annot=True, fmt='d', cmap='OrRd',
            xticklabels=['Normal', 'Attack'],
            yticklabels=['Normal', 'Attack'])
plt.title(f'Confusion Matrix — Condition 2: Under Attack (std={DEFENSE_NOISE_STD})')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion-matrix-attack.png', dpi=150)
plt.show()
print('Saved: confusion-matrix-attack.png')

In [ ]:
# ─── Cell 8: Results summary — copy to attack-results.md ──────────────────────
print('╔══════════════════════════════════════════════════════════════╗')
print('║      WEEK 5 RESULTS — COPY THESE TO attack-results.md       ║')
print('╠══════════════════════════════════════════════════════════════╣')
print(f'║  C1 Baseline: Accuracy={acc_c1:.4f} Recall={rec_c1:.4f} F1={f1_c1:.4f}  ║')
for std, m in attack_results.items():
    drop = rec_c1 - m['recall']
    print(f'║  std={std}: Accuracy={m["accuracy"]:.4f} Recall={m["recall"]:.4f} F1={m["f1"]:.4f} (drop:{drop:+.4f}) ║')
print(f'║  Selected defense level: DEFENSE_NOISE_STD = {DEFENSE_NOISE_STD}              ║')
print(f'║  C2 at selected level: Accuracy={acc_c2:.4f} Recall={rec_c2:.4f} F1={f1_c2:.4f} ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  Variables to keep for Week 6:                              ║')
print('║    DEFENSE_NOISE_STD, X_adv_selected, y_pred_adv_selected  ║')
print('║    acc_c2, rec_c2, f1_c2, auc_c2                           ║')
print('╚══════════════════════════════════════════════════════════════╝')